In [1]:
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

print(
    "Embedding dimensions:",
    embedding_model.get_sentence_embedding_dimension()
)

print(
    "Maximum input length:",
    embedding_model.max_seq_length
)

c:\Users\clarkb\Documents\movie-recommendation-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3067.91it/s]


Embedding dimensions: 384
Maximum input length: 256


C:\Users\clarkb\AppData\Local\Temp\ipykernel_29960\1048673858.py:13: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [3]:
example_texts = [
    "A detective investigates a mysterious disappearance.",
    "An investigator tries to find a missing person.",
    "A chef opens a restaurant and prepares dinner."
]

example_embeddings = embedding_model.encode(
    example_texts,
    normalize_embeddings=True
)

print("Embedding matrix shape:", example_embeddings.shape)

example_similarities = pd.DataFrame(
    cosine_similarity(example_embeddings),
    index=["Detective", "Missing person", "Chef"],
    columns=["Detective", "Missing person", "Chef"]
)

example_similarities

Embedding matrix shape: (3, 384)


,Detective,Missing person,Chef
Detective,1.000000,0.770500,0.050921
Missing person,0.770500,1.000000,0.053552
Chef,0.050921,0.053552,1.000000


In [4]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

training_plot_catalog = pd.read_csv(
    project_root / "data" / "processed" / "training_plot_catalog.csv",
    dtype={"wikipedia_movie_id": "Int64"}
)

training_plot_catalog["has_plot"] = (
    training_plot_catalog["plot"]
    .fillna("")
    .str.strip()
    .ne("")
)

text_movies = (
    training_plot_catalog.loc[
        training_plot_catalog["has_plot"]
    ]
    .sort_values("movieId")
    .reset_index(drop=True)
    .copy()
)

print("Training candidates:", len(training_plot_catalog))
print("Movies with plots:", len(text_movies))

Training candidates: 6233
Movies with plots: 5036


In [5]:
def count_plot_tokens(plot):
    encoded = embedding_model.tokenizer(
        plot,
        add_special_tokens=True,
        truncation=False,
        verbose=False
    )
    return len(encoded["input_ids"])


text_movies["token_count"] = text_movies["plot"].map(
    count_plot_tokens
)

input_limit = embedding_model.max_seq_length

exceeds_limit = text_movies["token_count"] > input_limit

print("Model input limit:", input_limit)
print("Plots exceeding the limit:", exceeds_limit.sum())
print("Percentage exceeding the limit:", f"{exceeds_limit.mean():.1%}")

text_movies["token_count"].describe()

Model input limit: 256
Plots exceeding the limit: 4264
Percentage exceeding the limit: 84.7%


count    5036.000000
mean      724.418983
std       419.260142
min        27.000000
25%       410.750000
50%       731.000000
75%       962.000000
max      5341.000000
Name: token_count, dtype: float64

84.7% of plots exceed the model’s limit, and the median plot contains 731 tokens. Embedding each full plot directly would therefore discard substantial text.

We’ll split plots into non-overlapping token chunks, embed each chunk, and later combine those embeddings into one vector per movie.

Calculate how much plot text fits in each chunk

In [6]:
tokenizer = embedding_model.tokenizer

special_token_count = tokenizer.num_special_tokens_to_add(pair=False)

content_token_limit = (
    embedding_model.max_seq_length - special_token_count
)

print("Special tokens per chunk:", special_token_count)
print("Plot tokens allowed per chunk:", content_token_limit)

Special tokens per chunk: 2
Plot tokens allowed per chunk: 254


In [7]:
chunk_rows = []

for movie in text_movies.itertuples(index=False):
    token_ids = tokenizer(
        movie.plot,
        add_special_tokens=False,
        truncation=False,
        verbose=False
    )["input_ids"]

    for chunk_number, start in enumerate(
        range(0, len(token_ids), content_token_limit),
        start=1
    ):
        chunk_token_ids = token_ids[
            start:start + content_token_limit
        ]

        chunk_rows.append({
            "movieId": movie.movieId,
            "chunk_number": chunk_number,
            "content_token_count": len(chunk_token_ids),
            "token_ids": chunk_token_ids
        })

plot_chunks = pd.DataFrame(chunk_rows)

In [8]:
assert (
    plot_chunks["content_token_count"]
    .between(1, content_token_limit)
    .all()
)

tokens_per_movie = plot_chunks.groupby("movieId")[
    "content_token_count"
].sum()

expected_tokens = (
    text_movies.set_index("movieId")["token_count"]
    - special_token_count
)

assert tokens_per_movie.reindex(
    expected_tokens.index
).eq(expected_tokens).all()

chunks_per_movie = plot_chunks.groupby("movieId").size()

print("Movies represented:", len(chunks_per_movie))
print("Total chunks:", len(plot_chunks))
print("Fewest chunks per movie:", chunks_per_movie.min())
print("Most chunks per movie:", chunks_per_movie.max())

display(
    plot_chunks[
        ["movieId", "chunk_number", "content_token_count"]
    ].head(8)
)

Movies represented: 5036
Total chunks: 16831
Fewest chunks per movie: 1
Most chunks per movie: 22


,movieId,chunk_number,content_token_count
0,1,1,254
1,1,2,254
2,1,3,254
3,1,4,79
4,2,1,254
5,2,2,254
6,2,3,254
7,2,4,85


In [10]:
import numpy as np
import torch
import torch.nn.functional as F

small_chunks = plot_chunks.head(4)

batch_token_ids = [
    [tokenizer.cls_token_id]
    + token_ids
    + [tokenizer.sep_token_id]
    for token_ids in small_chunks["token_ids"]
]

batch_length = max(len(ids) for ids in batch_token_ids)

assert batch_length <= embedding_model.max_seq_length

input_ids = torch.full(
    (len(batch_token_ids), batch_length),
    fill_value=tokenizer.pad_token_id,
    dtype=torch.long
)

attention_mask = torch.zeros_like(input_ids)

for row_number, token_ids in enumerate(batch_token_ids):
    length = len(token_ids)

    input_ids[row_number, :length] = torch.tensor(
        token_ids,
        dtype=torch.long
    )
    attention_mask[row_number, :length] = 1

batch_inputs = {
    "input_ids": input_ids,
    "attention_mask": attention_mask,
    "token_type_ids": torch.zeros_like(input_ids)
}

print("Input matrix shape:", tuple(input_ids.shape))
print(
    "Non-padding tokens per chunk:",
    attention_mask.sum(dim=1).tolist()
)

Input matrix shape: (4, 256)
Non-padding tokens per chunk: [256, 256, 256, 81]


In [11]:
batch_inputs = {
    name: tensor.to(embedding_model.device)
    for name, tensor in batch_inputs.items()
}

embedding_model.eval()

with torch.inference_mode():
    model_output = embedding_model(batch_inputs)

    normalized_embeddings = F.normalize(
        model_output["sentence_embedding"],
        p=2,
        dim=1
    )

    small_chunk_embeddings = normalized_embeddings.cpu().numpy()

print("Embedding matrix shape:", small_chunk_embeddings.shape)

print(
    "Embedding L2 norms:",
    np.linalg.norm(small_chunk_embeddings, axis=1)
)

print(
    "All values finite:",
    np.isfinite(small_chunk_embeddings).all()
)

Embedding matrix shape: (4, 384)
Embedding L2 norms: [1.         1.         0.99999994 1.        ]
All values finite: True


In [12]:
def embed_chunk_batch(token_id_lists, model):
    """Embed token chunks using our BERT-style model."""
    batch_tokenizer = model.tokenizer

    sequences = [
        [batch_tokenizer.cls_token_id]
        + token_ids
        + [batch_tokenizer.sep_token_id]
        for token_ids in token_id_lists
    ]

    batch_length = max(len(ids) for ids in sequences)

    assert batch_length <= model.max_seq_length

    input_ids = torch.full(
        (len(sequences), batch_length),
        fill_value=batch_tokenizer.pad_token_id,
        dtype=torch.long,
        device=model.device
    )

    attention_mask = torch.zeros_like(input_ids)

    for row_number, token_ids in enumerate(sequences):
        length = len(token_ids)

        input_ids[row_number, :length] = torch.tensor(
            token_ids,
            dtype=torch.long,
            device=model.device
        )
        attention_mask[row_number, :length] = 1

    features = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "token_type_ids": torch.zeros_like(input_ids)
    }

    model.eval()

    with torch.inference_mode():
        output = model(features)

        embeddings = F.normalize(
            output["sentence_embedding"],
            p=2,
            dim=1
        )

        return embeddings.cpu().numpy()

In [14]:
from tqdm import tqdm

batch_size = 16
embedding_dimensions = embedding_model.get_embedding_dimension()

chunk_embeddings = np.empty(
    (len(plot_chunks), embedding_dimensions),
    dtype=np.float32
)

for start in tqdm(
    range(0, len(plot_chunks), batch_size),
    desc="Embedding plot chunks",
    unit="batch"
):
    stop = min(start + batch_size, len(plot_chunks))

    token_id_lists = (
        plot_chunks["token_ids"]
        .iloc[start:stop]
        .tolist()
    )

    chunk_embeddings[start:stop] = embed_chunk_batch(
        token_id_lists,
        embedding_model
    )

Embedding plot chunks: 100%|██████████| 1052/1052 [06:19<00:00,  2.77batch/s]


In [15]:
chunk_norms = np.linalg.norm(chunk_embeddings, axis=1)

assert np.isfinite(chunk_embeddings).all()
assert np.allclose(chunk_norms, 1.0, atol=1e-5)

print("Chunk embedding shape:", chunk_embeddings.shape)
print("Smallest L2 norm:", chunk_norms.min())
print("Largest L2 norm:", chunk_norms.max())

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

np.savez_compressed(
    processed_dir / "plot_chunk_embeddings.npz",
    embeddings=chunk_embeddings,
    movie_ids=plot_chunks["movieId"].to_numpy(),
    chunk_numbers=plot_chunks["chunk_number"].to_numpy(),
    content_token_counts=plot_chunks["content_token_count"].to_numpy()
)

print("Saved plot_chunk_embeddings.npz")

Chunk embedding shape: (16831, 384)
Smallest L2 norm: 0.9999999
Largest L2 norm: 1.0000001
Saved plot_chunk_embeddings.npz


All 16,831 chunks are embedded, and the tiny differences from 1 are normal floating-point rounding.

Next, we combine each movie’s chunks into one 384-dimensional vector. We’ll weight chunks by their plot-token counts, so a short final chunk contributes less than a full chunk, then normalize the result.

In [16]:
# Positions of each movie's chunks in chunk_embeddings
chunk_positions = plot_chunks.groupby("movieId", sort=False).indices
chunk_lengths = plot_chunks["content_token_count"].to_numpy()

assert len(plot_chunks) == len(chunk_embeddings)
assert text_movies["movieId"].is_unique

movie_embeddings = np.empty(
    (len(text_movies), chunk_embeddings.shape[1]),
    dtype=np.float32
)

# Preserve the exact row order of text_movies
for movie_row, movie_id in enumerate(text_movies["movieId"]):
    positions = chunk_positions[movie_id]

    movie_embeddings[movie_row] = np.average(
        chunk_embeddings[positions],
        axis=0,
        weights=chunk_lengths[positions]
    )

# Check the averages before normalizing
movie_norms = np.linalg.norm(movie_embeddings, axis=1)

assert np.isfinite(movie_embeddings).all()
assert (movie_norms > 0).all()

# Give every movie vector unit length
movie_embeddings /= movie_norms[:, None]

In [17]:
final_norms = np.linalg.norm(movie_embeddings, axis=1)

assert np.isfinite(movie_embeddings).all()
assert np.allclose(final_norms, 1.0, atol=1e-5)

print("Movie embedding shape:", movie_embeddings.shape)
print("Smallest L2 norm:", final_norms.min())
print("Largest L2 norm:", final_norms.max())

np.savez_compressed(
    processed_dir / "movie_plot_embeddings.npz",
    embeddings=movie_embeddings,
    movie_ids=text_movies["movieId"].to_numpy()
)

print("Saved movie_plot_embeddings.npz")

Movie embedding shape: (5036, 384)
Smallest L2 norm: 0.9999999
Largest L2 norm: 1.0000001
Saved movie_plot_embeddings.npz


We now have one normalized 384-dimensional vector for each of the 5,036 movies.

Let’s repeat the Toy Story similarity check. Because the vectors have unit length, their dot product equals cosine similarity.

In [18]:
# Locate Toy Story's position in the embedding matrix
toy_story_position = np.flatnonzero(
    text_movies["movieId"].to_numpy() == 1
)[0]

# Compare Toy Story with every movie
similarities = (
    movie_embeddings @ movie_embeddings[toy_story_position]
)

print("Similarity scores calculated:", len(similarities))
print(
    "Toy Story similarity to itself:",
    similarities[toy_story_position]
)

similarity_results = text_movies[["movieId", "title"]].copy()
similarity_results["similarity"] = similarities

# Exclude Toy Story itself and rank the remaining movies
ranked_neighbors = (
    similarity_results
    .loc[similarity_results["movieId"] != 1]
    .sort_values(
        ["similarity", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

display(ranked_neighbors.head(10))

Similarity scores calculated: 5036
Toy Story similarity to itself: 1.0


,movieId,title,similarity
0,3114,Toy Story 2 (1999),0.808835
1,1992,Child's Play 2 (1990),0.576782
2,4990,Jimmy Neutron: Boy Genius (2001),0.571075
3,3699,Starman (1984),0.567056
4,2450,Howard the Duck (1986),0.563635
5,2804,"Christmas Story, A (1983)",0.560568
6,4544,Short Circuit 2 (1988),0.541734
7,4370,A.I. Artificial Intelligence (2001),0.528468
8,1592,Air Bud (1997),0.521333
9,1359,Jingle All the Way (1996),0.518031


In [19]:
comparison_movie_ids = [
    3114,  # Toy Story 2
    2041,  # Condorman
    318    # The Shawshank Redemption
]

display(
    ranked_neighbors.loc[
        ranked_neighbors["movieId"].isin(comparison_movie_ids)
    ]
)

,movieId,title,similarity
0,3114,Toy Story 2 (1999),0.808835
185,318,"Shawshank Redemption, The (1994)",0.423592
217,2041,Condorman (1981),0.416339


The ranking changed in a useful way:
Movie	TF-IDF rank	Embedding rank
Toy Story 2	1	1
The Shawshank Redemption	8	186
Condorman	2	218


Condorman and Shawshank now rank much lower, although their numerical similarities increased. The two representations produce different score distributions, so compare their rankings and evaluation results rather than interpreting a higher score as an improvement.

Next, let’s generate personalized recommendations using the same rules as TF-IDF:
- Use the user’s training ratings of 4 or higher.
- Average similarity to their liked movies that have plot embeddings.
- Exclude every movie they rated in training.
- Break ties using training popularity, then movie ID.
- Use popularity when no liked movies have plot features.
First, run your existing data-loading cells in this notebook so train_ratings contains the same training split used previously. Check:

In [22]:
from pathlib import Path

import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_dir = project_root / "data" / "raw" / "ml-latest-small"

movies = pd.read_csv(data_dir / "movies.csv")
ratings = pd.read_csv(data_dir / "ratings.csv")

ratings["rated_at"] = pd.to_datetime(
    ratings["timestamp"],
    unit="s",
    utc=True
)

dates = ratings["rated_at"]

train_cutoff = dates.quantile(0.60)
validation_cutoff = dates.quantile(0.80)

train_ratings = ratings.loc[
    dates <= train_cutoff
].copy()

validation_ratings = ratings.loc[
    (dates > train_cutoff) & (dates <= validation_cutoff)
].copy()

test_ratings = ratings.loc[
    dates > validation_cutoff
].copy()

print("Training rows:", len(train_ratings))
print("Validation rows:", len(validation_ratings))
print("Test rows:", len(test_ratings))

print("Training ratings:", len(train_ratings))
print("Training candidate movies:", train_ratings["movieId"].nunique())

Training rows: 60502
Validation rows: 20167
Test rows: 20167
Training ratings: 60502
Training candidate movies: 6233


In [23]:
def recommend_embedding_based(
    user_id,
    training_data,
    text_movies,
    movie_embeddings,
    candidate_catalog,
    n=10
):
    history = training_data.loc[
        training_data["userId"] == user_id
    ]

    liked_ids = history.loc[
        history["rating"] >= 4.0,
        "movieId"
    ]

    liked_mask = text_movies["movieId"].isin(liked_ids).to_numpy()

    recommendations = candidate_catalog.loc[
        ~candidate_catalog["movieId"].isin(history["movieId"]),
        ["movieId", "title", "has_plot"]
    ].copy()

    recommendations["score"] = 0.0

    if liked_mask.any():
        # Average the vectors of the user's liked movies
        user_profile = movie_embeddings[liked_mask].mean(axis=0)

        # Equals average cosine similarity to those liked movies
        movie_scores = movie_embeddings @ user_profile

        scores_by_id = pd.Series(
            movie_scores,
            index=text_movies["movieId"]
        )

        recommendations["score"] = (
            recommendations["movieId"]
            .map(scores_by_id)
            .fillna(0.0)
        )

    training_counts = training_data["movieId"].value_counts()

    recommendations["training_rating_count"] = (
        recommendations["movieId"]
        .map(training_counts)
        .fillna(0)
        .astype(int)
    )

    return (
        recommendations
        .sort_values(
            ["score", "training_rating_count", "movieId"],
            ascending=[False, False, True]
        )
        .head(n)
        .reset_index(drop=True)
    )

The average vector is a user preference profile. We leave that average unnormalized so the dot product remains exactly the average cosine similarity to the liked movies.

As in your TF-IDF implementation, movies without plots receive zero. Embedding similarities can be negative, so those zero-score candidates can rank above movies with negative scores.

Now check users 1 and 2:

In [24]:
for user_id in [1, 2]:
    print(f"\nRecommendations for user: {user_id}")

    display(
        recommend_embedding_based(
            user_id=user_id,
            training_data=train_ratings,
            text_movies=text_movies,
            movie_embeddings=movie_embeddings,
            candidate_catalog=training_plot_catalog,
            n=10
        )
    )


Recommendations for user: 1


,movieId,title,has_plot,score,training_rating_count
0,3262,Twin Peaks: Fire Walk with Me (1992),True,0.444358,7
1,3837,Phantasm II (1988),True,0.442189,3
2,2799,Problem Child 2 (1991),True,0.441442,7
3,3442,Band of the Hand (1986),True,0.437013,1
4,2535,Earthquake (1974),True,0.436511,3
5,3017,Creepshow 2 (1987),True,0.435831,4
6,33166,Crash (2004),True,0.435309,31
7,3016,Creepshow (1982),True,0.434908,11
8,48518,"Texas Chainsaw Massacre: The Beginning, The (2...",True,0.433899,1
9,2878,Hell Night (1981),True,0.433380,1



Recommendations for user: 2


,movieId,title,has_plot,score,training_rating_count
0,296,Pulp Fiction (1994),True,0.0,201
1,356,Forrest Gump (1994),True,0.0,193
2,480,Jurassic Park (1993),True,0.0,185
3,318,"Shawshank Redemption, The (1994)",True,0.0,177
4,593,"Silence of the Lambs, The (1991)",True,0.0,177
5,110,Braveheart (1995),True,0.0,174
6,589,Terminator 2: Judgment Day (1991),True,0.0,166
7,457,"Fugitive, The (1993)",True,0.0,160
8,592,Batman (1989),True,0.0,160
9,780,Independence Day (a.k.a. ID4) (1996),True,0.0,157


User 2’s output matches the popularity fallback. User 1 now receives embedding-based scores, but those scores are similarities, not probabilities of liking a movie.

Let’s evaluate Recall@10 using the same validation split. Keep all positively rated validation movies in the denominator, including movies unavailable in the training catalog or without plots.

Make sure validation_ratings is loaded from your existing split, then run:

In [25]:
validation_positives = validation_ratings.loc[
    validation_ratings["rating"] >= 4.0
]

training_users = set(train_ratings["userId"])
evaluation_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    recommendations = recommend_embedding_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        movie_embeddings=movie_embeddings,
        candidate_catalog=training_plot_catalog,
        n=10
    )

    positive_movie_ids = set(user_positives["movieId"])
    recommended_movie_ids = set(recommendations["movieId"])

    hits = len(
        positive_movie_ids.intersection(recommended_movie_ids)
    )

    evaluation_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_movie_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_movie_ids),
        "has_training_history": user_id in training_users
    })

embedding_results = pd.DataFrame(evaluation_rows)

print("Users evaluated:", len(embedding_results))
print(
    "Embedding mean Recall@10:",
    embedding_results["recall_at_10"].mean()
)

display(
    embedding_results
    .groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

Users evaluated: 165
Embedding mean Recall@10: 0.03561627154195268


,users,mean_recall_at_10
has_training_history,,
False,140,0.041847
True,25,0.000724


he embedding recommender performed worse than TF-IDF on this validation split.
Model	Overall Recall@10	Recall@10 with training history
Original TF-IDF	3.761%	1.387%
Log-TF TF-IDF	3.696%	0.957%
Semantic embeddings	3.562%	0.072%


The overall decline looks modest because 140 of 165 users receive the unchanged popularity fallback. Among the 25 users with training history, the decline is substantial.

Our Toy Story example showed more plausible item similarities, but personalized recommendations involve another step: averaging across a user’s liked movies. A broad average may favor movies that are moderately similar to many plots. That is a hypothesis we can investigate.

Let’s check whether the same movies repeatedly appear across users’ recommendation lists:

In [26]:
warm_results = embedding_results.loc[
    embedding_results["has_training_history"]
].copy()

print("Users with training history:", len(warm_results))
print(
    "Users with at least one hit:",
    warm_results["hits_at_10"].gt(0).sum()
)
print("Total hits:", warm_results["hits_at_10"].sum())

recommendation_tables = []

for user_id in warm_results["userId"]:
    recommendations = recommend_embedding_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        movie_embeddings=movie_embeddings,
        candidate_catalog=training_plot_catalog,
        n=10
    )

    recommendations["userId"] = user_id
    recommendation_tables.append(recommendations)

warm_recommendations = pd.concat(
    recommendation_tables,
    ignore_index=True
)

recommendation_frequency = (
    warm_recommendations
    .groupby(["movieId", "title"], as_index=False)
    .agg(
        users_recommended_to=("userId", "nunique")
    )
    .sort_values(
        ["users_recommended_to", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print(
    "Distinct movies across these recommendation lists:",
    warm_recommendations["movieId"].nunique()
)

display(recommendation_frequency.head(15))

Users with training history: 25
Users with at least one hit: 3
Total hits: 3
Distinct movies across these recommendation lists: 50


,movieId,title,users_recommended_to
0,2535,Earthquake (1974),22
1,3262,Twin Peaks: Fire Walk with Me (1992),20
2,3433,Death Wish 4: The Crackdown (1987),20
3,2799,Problem Child 2 (1991),19
4,3837,Phantasm II (1988),19
5,2338,I Still Know What You Did Last Summer (1998),13
6,33166,Crash (2004),12
7,8369,Mindhunters (2004),11
8,1976,Friday the 13th Part 3: 3D (1982),8
9,3442,Band of the Hand (1986),8


The recommendations are highly concentrated: the five most repeated movies occupy 100 of the 250 recommendation slots—40%. Earthquake appears for 22 of 25 users. Combined with only three hits, this warrants investigating what drives their scores.

One possibility is that these movies score highly against many plots generally, so they repeatedly rise to the top when we average similarities across a user’s liked movies.

Let’s test that directly: calculate each movie’s average similarity to the entire plot catalog.

In [27]:
# Average vector across all movies with plot features
catalog_mean_vector = movie_embeddings.mean(axis=0)

# Each movie's average cosine similarity to the plot catalog
catalog_mean_similarities = (
    movie_embeddings @ catalog_mean_vector
)

catalog_similarity = text_movies[["movieId", "title"]].copy()

catalog_similarity["mean_catalog_similarity"] = (
    catalog_mean_similarities
)

catalog_similarity = (
    catalog_similarity
    .sort_values(
        ["mean_catalog_similarity", "movieId"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

catalog_similarity["catalog_similarity_rank"] = (
    np.arange(1, len(catalog_similarity) + 1)
)

# Compare general catalog similarity with recommendation frequency
frequency_diagnostic = recommendation_frequency.merge(
    catalog_similarity[
        [
            "movieId",
            "mean_catalog_similarity",
            "catalog_similarity_rank"
        ]
    ],
    on="movieId",
    how="left",
    validate="one_to_one"
)

display(
    frequency_diagnostic
    .sort_values(
        ["users_recommended_to", "movieId"],
        ascending=[False, True]
    )
    .head(15)
)

,movieId,title,users_recommended_to,mean_catalog_similarity,catalog_similarity_rank
0,2535,Earthquake (1974),22,0.434943,2
1,3262,Twin Peaks: Fire Walk with Me (1992),20,0.442230,1
2,3433,Death Wish 4: The Crackdown (1987),20,0.431357,5
3,2799,Problem Child 2 (1991),19,0.433648,3
4,3837,Phantasm II (1988),19,0.422810,9
5,2338,I Still Know What You Did Last Summer (1998),13,0.428256,6
6,33166,Crash (2004),12,0.433316,4
7,8369,Mindhunters (2004),11,0.416482,15
8,1976,Friday the 13th Part 3: 3D (1982),8,0.416682,14
9,3442,Band of the Hand (1986),8,0.419421,11


This strongly supports the catalog-similarity explanation: all 15 frequently recommended movies rank among the top 49 for similarity to the whole catalog.

Let’s test one adjustment:
\[
\text{adjusted score}
=
\text{average similarity to the user’s liked movies}
-
\text{average similarity to the catalog}
\]
For example, user 1’s score for Twin Peaks becomes approximately:
\[
0.444358 - 0.442230 = 0.002128
\]
Its high original score mostly overlaps with its catalog baseline. The adjusted score measures how much extra similarity it has to this user’s liked movies.

1. Add the adjustment to your existing function.

In the function-definition cell, replace its header with:

In [30]:
def recommend_embedding_based(
    user_id,
    training_data,
    text_movies,
    movie_embeddings,
    candidate_catalog,
    n=10,
    catalog_mean_vector=None
):
    history = training_data.loc[
        training_data["userId"] == user_id
    ]

    liked_ids = history.loc[
        history["rating"] >= 4.0,
        "movieId"
    ]

    liked_mask = text_movies["movieId"].isin(liked_ids).to_numpy()

    recommendations = candidate_catalog.loc[
        ~candidate_catalog["movieId"].isin(history["movieId"]),
        ["movieId", "title", "has_plot"]
    ].copy()

    recommendations["score"] = 0.0

    if liked_mask.any():
        # Average the vectors of the user's liked movies
        user_profile = movie_embeddings[liked_mask].mean(axis=0)
        
        if catalog_mean_vector is not None:
            user_profile = user_profile - catalog_mean_vector
        # Equals average cosine similarity to those liked movies
        movie_scores = movie_embeddings @ user_profile

        scores_by_id = pd.Series(
            movie_scores,
            index=text_movies["movieId"]
        )

        recommendations["score"] = (
            recommendations["movieId"]
            .map(scores_by_id)
            .fillna(0.0)
        )

    training_counts = training_data["movieId"].value_counts()

    recommendations["training_rating_count"] = (
        recommendations["movieId"]
        .map(training_counts)
        .fillna(0)
        .astype(int)
    )

    return (
        recommendations
        .sort_values(
            ["score", "training_rating_count", "movieId"],
            ascending=[False, False, True]
        )
        .head(n)
        .reset_index(drop=True)
    )

In [31]:
catalog_mean_vector = movie_embeddings.mean(axis=0)

adjusted_rows = []

for user_id, user_positives in validation_positives.groupby("userId"):
    recommendations = recommend_embedding_based(
        user_id=user_id,
        training_data=train_ratings,
        text_movies=text_movies,
        movie_embeddings=movie_embeddings,
        candidate_catalog=training_plot_catalog,
        n=10,
        catalog_mean_vector=catalog_mean_vector
    )

    positive_ids = set(user_positives["movieId"])
    recommended_ids = set(recommendations["movieId"])

    hits = len(positive_ids.intersection(recommended_ids))

    adjusted_rows.append({
        "userId": user_id,
        "positive_movie_count": len(positive_ids),
        "hits_at_10": hits,
        "recall_at_10": hits / len(positive_ids),
        "has_training_history": user_id in training_users
    })

adjusted_embedding_results = pd.DataFrame(adjusted_rows)

print("Users evaluated:", len(adjusted_embedding_results))

print(
    "Original embedding Recall@10:",
    embedding_results["recall_at_10"].mean()
)

print(
    "Adjusted embedding Recall@10:",
    adjusted_embedding_results["recall_at_10"].mean()
)

display(
    adjusted_embedding_results
    .groupby("has_training_history")
    .agg(
        users=("userId", "size"),
        mean_recall_at_10=("recall_at_10", "mean")
    )
)

Users evaluated: 165
Original embedding Recall@10: 0.03561627154195268
Adjusted embedding Recall@10: 0.035910199258694826


,users,mean_recall_at_10
has_training_history,,
False,140,0.041847
True,25,0.002664
